# Lesson 02 - Eksploracja Microsoft Agent Framework

The **Microsoft Agent Framework (MAF)** to zunifikowany framework do budowania agentów AI. Zapewnia czystą, składalną architekturę z czterema podstawowymi elementami:

- **Client** – łączy się z punktem końcowym modelu AI i obsługuje komunikację
- **Agent** – opakowuje klienta z instrukcjami i definicjami narzędzi
- **Tools** – rozszerzają możliwości agenta o niestandardowe funkcje, które model może wywoływać
- **Session** – utrzymuje historię rozmów dla interakcji wieloetapowych

W tej lekcji zbudujemy **agenta do rezerwacji podróży**, który sprawdza dostępność celu podróży, wykorzystując te koncepcje.


## Konfiguracja


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Zrozumienie architektury Agent Framework

Microsoft Agent Framework opiera się na architekturze warstwowej:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Klient** – `AzureAIProjectAgentProvider` łącze się z wdrożeniem Azure OpenAI. Obsługuje uwierzytelnianie, formatowanie zapytań i parsowanie odpowiedzi.
2. **Agent** – Tworzony z klienta za pomocą `provider.create_agent()`, agent łączy dostęp do modelu z instrukcjami (podpowiedź systemowa) oraz narzędziami.
3. **Narzędzia** – Funkcje Pythona dekorowane za pomocą `@tool`, które agent może wywoływać, aby wykonać działania lub pobrać dane.
4. **Sesja** – Obiekt `AgentSession` (tworzony za pomocą `agent.create_session()`) przechowujący historię rozmowy, umożliwiający dialog wielokrotnego przebiegu, gdzie agent pamięta wcześniejszy kontekst.

Zbudujmy każdą warstwę krok po kroku.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Dodawanie narzędzi za pomocą dekoratora @tool

Narzędzia pozwalają agentom podejmować działania wykraczające poza generowanie tekstu. Dekorator `@tool` zamienia zwykłą funkcję Pythona w coś, co agent może wywołać.

Kluczowe punkty:
- Używaj `Annotated[type, "description"]`, aby model rozumiał każdy parametr.
- Docstring staje się opisem narzędzia, które widzi model.
- `approval_mode="never_require"` oznacza, że narzędzie działa automatycznie bez potwierdzenia użytkownika.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Tworzenie Agenta z Narzędziami

Teraz łączymy klienta, instrukcje i narzędzia w agenta. `instructions` pełnią rolę podpowiedzi systemowej — definiują osobowość i zachowanie agenta.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Rozmowy wieloetapowe z sesjami

`AgentSession` (tworzona za pomocą `agent.create_session()`) śledzi wszystkie wiadomości w rozmowie. Przekazując tę samą sesję do każdego wywołania `agent.run()`, agent ma dostęp do pełnej historii konwersacji i może odwoływać się do wcześniejszych wiadomości.

Przekazujemy `tools=[check_destination_availability]`, aby agent mógł wywoływać nasz sprawdzacz dostępności podczas każdej tury.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Podsumowanie

W tej lekcji zapoznałeś się z czterema filarami Microsoft Agent Framework:

| Koncepcja | Czego się nauczyłeś |
|---------|------------------|
| **Klient** | `AzureAIProjectAgentProvider` łączy się z Azure OpenAI za pomocą uwierzytelniania opartego na poświadczeniach |
| **Agent** | `provider.create_agent()` łączy połączenie z modelem z instrukcjami i nazwą |
| **Narzędzia** | Dekorator `@tool` udostępnia funkcje Pythona, które agent może wywoływać |
| **Sesja** | `agent.create_session()` utrzymuje historię rozmowy przez wiele wymian |

Te elementy składowe łączą się, tworząc agentów, którzy potrafią prowadzić naturalne rozmowy, wywoływać funkcje zewnętrzne i utrzymywać kontekst — stanowią fundament do bardziej zaawansowanych wzorców agentowych w kolejnych lekcjach.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:  
Dokument ten został przetłumaczony za pomocą usługi tłumaczeń AI [Co-op Translator](https://github.com/Azure/co-op-translator). Chociaż dokładamy wszelkich starań, aby tłumaczenie było precyzyjne, prosimy pamiętać, że automatyczne tłumaczenia mogą zawierać błędy lub nieścisłości. Oryginalny dokument w języku źródłowym powinien być uznawany za wiążące źródło informacji. W przypadku istotnych informacji zaleca się skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z korzystania z tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
